# Tutor cluster: analysis

Sections follow the dashboard order: app → gateway decisions → placement → hops → workers → overflow → errors → goodput. Numbers only; the answers are in DESIGN.md.

In [ ]:
import json, statistics, sys
from collections import Counter
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "metrics").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "metrics"))
import summarise_run, goodput  # the same code the reports use

RUN = "run_i"            # the run to show: run_c ... run_i (gateway runs); baseline_a / baseline_b have no gateway log
RUN_DIR = ROOT / "metrics" / RUN
if not (RUN_DIR / "gateway_requests.jsonl").exists():
    RUN, RUN_DIR = "run_g", ROOT / "metrics" / "run_g"   # fall back to run G if the chosen run is missing


def load(pattern, folder=None):
    return [json.loads(l) for p in sorted((folder or RUN_DIR).glob(pattern)) for l in p.read_text().splitlines() if l.strip()]


gw = load("gateway_requests.jsonl")
traffic = load("traffic_*.jsonl")
S = summarise_run.main(RUN_DIR)
print(f"RUN = {RUN}: {len(traffic)} client requests, {len(gw)} gateway rows")


## 1. App: what the tutor sends

In [ ]:
ok = [r for r in traffic if r["status"] == 200]
for t in sorted({r["tenant"] for r in ok}):
    p = [r["prompt_tokens"] for r in ok if r["tenant"] == t and r.get("prompt_tokens")]
    o = [r["output_tokens"] for r in ok if r["tenant"] == t and r.get("output_tokens")]
    print(f"{t:11}: prompt tokens median {statistics.median(p):.0f}, output tokens median {statistics.median(o):.0f}, n {len(p)}")
print("shared prefix: the 431-token system prompt (DESIGN Part 1); chapters sent (X-Chapter):",
      dict(Counter(r.get("top_chapter") for r in traffic).most_common(10)))


## 2. Gateway decisions: guard, admit, sheds by reason and tenant

In [ ]:
print("decisions:", S["decisions"])
print("guard rejects:", S["guard_rejects"] or "none")
print("sheds by stage:", S["sheds_by_stage"])
print("sheds by tenant/reason:", S["sheds_by_reason_tenant"])
print("admitted by tenant:", S["admitted_by_tenant"])


## 3. Placement: WorkerA / WorkerB

In [ ]:
print("preferred (chapter home):", S["preferred_by_worker"])
print("chosen (placed):        ", S["placed_by_worker"])
print("policy / load bound:", {(r.get("policy"), r.get("load_bound")) for r in gw if r.get("policy")})


## 4. Hops

In [ ]:
print(json.dumps(S["hops"], indent=1))


## 5. Workers: in flight, waiting, KV, preemptions, prefix hit rate

In [ ]:
for w, v in S["workers"].items():
    print(w, {k: v.get(k) for k in ("gateway_in_flight_max", "gateway_in_flight_mean", "gateway_queue_max",
                                     "max_waiting", "max_kv_pct", "preemptions", "prefix_hit_pct", "engine_ttft_mean_ms")})


## 6. Overflow

In [ ]:
print(json.dumps(S.get("overflow", "no overflow in this run"), indent=1))
print("overflow rows by outcome:", dict(Counter(r.get("overflow_outcome") for r in gw if r.get("overflow_outcome"))))


## 7. Errors

In [ ]:
print("client statuses:", dict(Counter(str(r["status"]) for r in traffic)))
print("aborts:", S["aborts"])
print("upstream errors:", dict(Counter(r.get("outcome") for r in gw if str(r.get("outcome", "")).startswith(("upstream", "overflow_unavailable", "leave")))) or "none")


## 8. Goodput across runs

In [ ]:
rows = [r for r in (goodput.run_goodput(ROOT / "metrics" / f, l, u, g) for f, l, u, g in goodput.RUNS) if r]
print(f"{'run':4} {'users':>5} {'offered/min':>11} {'goodput/min':>11} {'share':>6}")
for r in rows:
    print(f"{r['run']:4} {r['users']:5} {r['offered_per_min']:11} {r['goodput_per_min']:11} {r['goodput_share']:6}")


## 9. Estimate vs measured TTFT, with and without the network trip

In [ ]:
print("network offset:", json.loads((ROOT / "metrics" / "network_offset.json").read_text())["offset_s"], "s")
for run in ("run_d", "run_e", "run_f", "run_g", "run_f2", "run_h1", "run_h2", "run_i"):
    if (ROOT / "metrics" / run / "gateway_requests.jsonl").exists():
        e = summarise_run.main(ROOT / "metrics" / run).get("est_vs_measured", {})
        print(f"{run:7} est p50 {e.get('est_p50_s')} s, measured p50 {e.get('measured_p50_s')} s, minus trip {e.get('measured_minus_trip_p50_s')} s; "
              f"within 50 %: {e.get('within_50pct')} raw, {e.get('within_50pct_minus_trip')} minus trip")


## Part 5: the nine scrape questions

Numbers only. Pavi's answers are in DESIGN.md Part 5.

### Q5.1. Who sits in your queue vs vLLM's waiting queue?

In [ ]:
for run in ("run_g", "run_i"):
    s = summarise_run.main(ROOT / "metrics" / run)
    print(run, {w: (v.get("gateway_queue_max"), v.get("max_waiting")) for w, v in s["workers"].items()}, "(gateway queue max, vLLM waiting max)")

### Q5.2. Waiting / running / swapped (or preempted)?

In [ ]:
s = summarise_run.main(ROOT / "metrics" / "run_f2")
print({w: {k: v.get(k) for k in ("max_running", "max_waiting", "preemptions", "max_kv_pct")} for w, v in s["workers"].items()})

### Q5.3. orch_replica_queue_depth (or your equivalent): which pod, under which mix?

In [ ]:
for run in ("run_d", "run_g", "run_h1", "run_h2", "run_i"):
    if (ROOT / "metrics" / run / "summary.json").exists() or (ROOT / "metrics" / run / "gateway_requests.jsonl").exists():
        s = summarise_run.main(ROOT / "metrics" / run)
        print(run, {w: v.get("gateway_queue_max") for w, v in s["workers"].items()}, {m: v.get("sent") for m, v in s["by_mix"].items()})

### Q5.4. If a 32k RAG retrieve and a short agent decode are both ready, who goes first: you or the engine?

In [ ]:
s = summarise_run.main(ROOT / "metrics" / "run_g")
print("gateway TTFT by tenant:", s["gateway_ttft_ms"])
src = (ROOT / "control" / "queue.py").read_text()
print([l for l in src.splitlines() if l.startswith("PRIORITY")])

### Q5.5. PagedAttention packs KV; prefix cache reuses KV. Which one saved memory on your shared-prefix mix?

In [ ]:
for run in ("run_h1", "run_h2", "run_f", "run_f2"):
    if (ROOT / "metrics" / run / "gateway_requests.jsonl").exists():
        s = summarise_run.main(ROOT / "metrics" / run)
        print(run, {w: (v.get("prefix_hit_pct"), v.get("max_kv_pct")) for w, v in s["workers"].items()}, "(prefix hit %, max KV %)")

### Q5.6. Chunked prefill and continuous batching: engine flags used and why?

In [ ]:
src = (ROOT / "cluster" / "modal_vllm.py").read_text()
print(src[src.index("ENGINE_ARGS = ["):src.index("]", src.index("ENGINE_ARGS = [")) + 1])

### Q5.7. KV full after admit: do you shed at the door, or does the engine preempt?

In [ ]:
s = summarise_run.main(ROOT / "metrics" / "run_f2")
print("kv_free sheds:", {k: v for k, v in s["sheds_by_reason_tenant"].items() if "kv_free" in k})
print("preemptions:", {w: v.get("preemptions") for w, v in s["workers"].items()})

### Q5.8. Client gone (aborted): who frees the KV? And how?

In [ ]:
print("aborts in each run:", {run: summarise_run.main(ROOT / "metrics" / run)["aborts"]
       for run in ("run_d", "run_e", "run_f", "run_g", "run_f2") if (ROOT / "metrics" / run).exists()})

### Q5.9. After a worker returns: slam it at 100 %, or ramp while p99 holds?

In [ ]:
print(json.dumps({w: [round(p["engine_ttft_s"] * 1000) for p in v["probes"]]
                  for w, v in json.loads((ROOT / "metrics" / "warm_proof.json").read_text())["workers"].items()}))